# Notebook 03 — Independent results review

Purpose: verify a completed three-arm semantic-local pilot from saved evidence. Inputs: source/config identities, audited split, full checkpoints, predictions and compute records. Outputs: `independent_review.json`; readable results and saved figures. Expected runtime: under two minutes. No training is launched. Checkpoints remain under `checkpoints/<RUN_ID>/`. Exact experiment resume is through the main Notebook 03 with the same RUN_ID; this review can simply be rerun top-to-bottom.

The review replays each selected linear classifier on its saved development features on CPU, checks all top-1/top-5 predictions, independently reconciles contrast point estimates and training-only normalization, and checks the frozen main notebook plus the recorded three-hour completion window. Bootstrap interval reproduction uses the same helper/algorithm as the original evaluator and is a consistency check. No encoder or GPU is used.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT=Path('/home/benjamin/workspace/ahr_go/fmm_jeppa')
_ns={'__file__':str(ROOT/'fmm_jepa/bootstrap.ipynb')}
for cell in nbformat.read(ROOT/'fmm_jepa/bootstrap.ipynb',as_version=4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
        exec(compile(cell.source,'bootstrap#ahr-export','exec'),_ns)
_ns['install_notebook_package'](ROOT)


In [ ]:
"""Read-only independent audit of a completed bounded semantic pilot."""
import csv, hashlib, json
import numpy as np
import torch
from torch.nn import functional as F
from fmm_jepa.training import definition_hashes, digest_file, atomic_json
from fmm_jepa.food101_training import validate_training_manifest
from fmm_jepa.food101_evaluation import paired_bootstrap


def reconcile_frozen_probe(feature_folder, probe_path, predictions, summary,
                           train_ids, dev_ids, heldout_ids):
    """Independently rebuild saved classifier predictions from CPU feature caches.

    No encoder is invoked, no labels fit a new classifier, and no held-out
    feature or prediction data are read. Training features verify that the
    stored normalization was fitted to precisely the registered training set.
    """
    feature_folder=Path(feature_folder);probe_path=Path(probe_path)
    selected=torch.load(probe_path,map_location='cpu',weights_only=True)
    assert selected['encoder_checkpoint_hash']==summary['checkpoint_hash']
    assert selected['probe_hash']==summary['probe_hash']
    assert selected['selected_epoch']==summary['selected_epoch']
    assert set(selected['state_dict'])=={'weight','bias'}
    caches={};evidence={}
    for split,ids in (('train',list(train_ids)),('development',list(dev_ids))):
        matches=[]
        for manifest_path in feature_folder.glob('*.json'):
            metadata=json.loads(manifest_path.read_text())
            if metadata.get('identity',{}).get('sample_ids')==ids:
                matches.append((manifest_path,metadata))
        assert len(matches)==1, f'Expected exactly one {split} cache: {feature_folder}'
        manifest_path,metadata=matches[0];cache_path=manifest_path.with_suffix('.pt')
        assert digest_file(cache_path)==metadata['sha256'], 'Feature file hash mismatch'
        cache=torch.load(cache_path,map_location='cpu',weights_only=True)
        assert cache['identity']==metadata['identity']
        assert cache['sample_ids']==ids and cache['identity']['sample_ids']==ids
        assert len(set(ids))==len(ids)
        assert not set(ids)&set(heldout_ids), 'Held-out sample in probe feature cache'
        assert cache['identity']['checkpoint_hash']==summary['checkpoint_hash']
        identity_key='train_identity' if split=='train' else 'dev_identity'
        content_key='train_cache' if split=='train' else 'dev_cache'
        assert selected['identity'][identity_key]==cache['identity']
        assert selected['identity'][content_key]==cache['content_hash']
        features,labels=cache['features'],cache['labels']
        assert features.device.type=='cpu' and labels.device.type=='cpu'
        assert features.ndim==2 and labels.ndim==1 and len(features)==len(labels)==len(ids)
        assert features.shape[1]==selected['input_dim']
        assert torch.isfinite(features).all() and labels.dtype==torch.long
        assert bool(((labels>=0)&(labels<101)).all())
        caches[split]=cache
        evidence[split]={'file':str(cache_path),'sha256':metadata['sha256'],'examples':len(ids)}
    stats=selected['standardization'];train=caches['train']['features'].float()
    if stats['enabled']:
        expected_mean=train.mean(0)
        expected_std=train.std(0,unbiased=False).clamp_min(1e-6)
    else:
        expected_mean=torch.zeros(train.shape[1]);expected_std=torch.ones(train.shape[1])
    # Reductions may group differently across CPU thread counts; require
    # numerical agreement while the saved statistics reproduce the exact probe.
    assert torch.allclose(stats['mean'],expected_mean,rtol=1e-6,atol=1e-7)
    assert torch.allclose(stats['std'],expected_std,rtol=1e-6,atol=1e-7)
    assert stats['fit_examples']==len(train_ids)
    weight,bias=selected['state_dict']['weight'],selected['state_dict']['bias']
    assert weight.device.type==bias.device.type=='cpu'
    assert weight.shape==(101,selected['input_dim']) and bias.shape==(101,)
    assert torch.isfinite(weight).all() and torch.isfinite(bias).all()
    features=caches['development']['features'].float()
    labels=caches['development']['labels']
    normalized=(features-stats['mean'])/stats['std']
    batch_size=int(selected['config']['batch_size'])
    with torch.inference_mode():
        logits=torch.cat([F.linear(part,weight,bias) for part in normalized.split(batch_size)])
    assert logits.shape==(len(dev_ids),101) and torch.isfinite(logits).all()
    top=logits.topk(5,dim=1).indices
    assert [row['sample_id'] for row in predictions]==list(dev_ids), 'Prediction order/membership mismatch'
    assert len(predictions)==len(dev_ids)
    for index,row in enumerate(predictions):
        assert row['split']=='development' and row['model_id']==summary['model_id']
        assert row['checkpoint_hash']==summary['checkpoint_hash'] and row['probe_hash']==summary['probe_hash']
        assert row['label']==int(labels[index])
        assert row['top1_prediction']==int(top[index,0])
        assert row['top5_predictions']==top[index].tolist()
        assert row['correct_top1']==int(top[index,0]==labels[index])
        assert row['correct_top5']==int((top[index]==labels[index]).any())
    top1=float((top[:,0]==labels).double().mean())
    top5=float((top==labels[:,None]).any(1).double().mean())
    loss=float(F.cross_entropy(logits,labels))
    assert np.isclose(top1,summary['metrics']['top1'],rtol=0,atol=1e-7)
    assert np.isclose(top5,summary['metrics']['top5'],rtol=0,atol=1e-7)
    assert np.isclose(loss,summary['metrics']['loss'],rtol=1e-6,atol=1e-7)
    return {'status':'PASS','probe_path':str(probe_path),'probe_sha256':digest_file(probe_path),
            'feature_caches':evidence,'device':'cpu','examples':len(dev_ids),
            'top1':top1,'top5':top5,'loss':loss,'all_top1_and_top5_predictions_equal':True,
            'training_only_normalization_recomputed':True,'heldout_features_or_predictions_read':False}


def audit_semantic_run(root, run_id):
    root=Path(root);folder=root/'artifacts/notebook_03'/run_id
    read=lambda name:json.loads((folder/name).read_text())
    config=read('resolved_config.json');protocol=read('protocol.json')
    training=read('training_summary.json');evaluation=read('semantic_evaluation.json')
    compute=read('semantic_compute_benchmark.json');clock=read('budget_clock.json')
    completion=read('pilot_status.json')
    authorized_seconds=float(clock['deadline_unix'])-float(clock['started_unix'])
    assert 0 < authorized_seconds <= 10800 + 1e-6, 'Run allowance exceeds three hours'
    assert completion['status']=='COMPLETED', 'Pilot has not recorded completion'
    assert float(clock['started_unix']) <= float(completion['updated_unix']) <= float(clock['deadline_unix']), 'Completion outside the authorized run window'
    assert 0 <= float(completion['elapsed_seconds']) <= authorized_seconds, 'Recorded elapsed time exceeds allowance'
    main=json.loads((root/'notebooks/03_Semantic_Local_Attention_Pilot.ipynb').read_text())
    consumer_sources=[''.join(cell['source']) for cell in main['cells'] if cell['cell_type']=='code']
    consumer_hash=hashlib.sha256(json.dumps(consumer_sources,sort_keys=True,separators=(',',':')).encode()).hexdigest()
    assert consumer_hash==protocol['consumer_notebook_sha256'], 'Main consumer notebook source changed' 
    assert evaluation['status']=='COMPLETED' and len(evaluation['probes'])==6
    assert training['equal_updates'] and training['paired_trace_equal']
    assert protocol['config']==config and protocol['source_hashes']==definition_hashes(root)
    check={};kinds=('vit','fmm','semantic')
    check['source_consumer_and_time_budget']={'status':'PASS','consumer_sha256':consumer_hash,
        'authorized_seconds':authorized_seconds,'recorded_elapsed_seconds':completion['elapsed_seconds'],
        'completed_unix':completion['updated_unix'],'deadline_unix':clock['deadline_unix'],
        'scope':clock['scope']}
    original=json.loads((root/'artifacts/notebook_03/implementation_review/baseline_source_hashes.json').read_text())
    for name,digest in original['sha256'].items():
        assert digest_file(root/name)==digest, 'Predecessor source changed: '+name
    check['predecessor_sources_unchanged']={'status':'PASS','modules':len(original['sha256'])}
    data=read('data_reuse_evidence.json')
    assert data['verified_selected_images']==12120 and data['test_pixels_read']==0
    assert evaluation['heldout_status']=='NOT_RUN'
    split=read('split_manifest.json')['selected_ids']
    assert len(split['training'])==10100 and len(split['development'])==2020
    assert not set(split['training']) & set(split['development'])
    assert not (set(split['training'])|set(split['development'])) & set(split['heldout'])
    check['split_and_data_integrity']={'status':'PASS','train':10100,'development':2020,'test_used':False}
    expected_updates=evaluation['actual_successful_updates']
    effective=config['training']['microbatch']*config['training']['accumulation_steps']
    trace_digests={};performances={};counts=read('parameter_counts.json')
    initial=read('initialization/manifest.json')
    for kind in kinds:
        directory=root/'checkpoints'/run_id/kind
        manifest=validate_training_manifest(directory)
        assert manifest['successful_updates']==expected_updates
        assert manifest['attempted_updates']==expected_updates
        assert manifest['image_exposures']==effective*expected_updates
        assert manifest['resume_notebook']=='notebooks/03_Semantic_Local_Attention_Pilot.ipynb'
        saved=torch.load(directory/manifest['latest'],map_location='cpu',weights_only=False)
        assert saved['scheduler']['position']==expected_updates and saved['ema_position']==expected_updates
        assert saved['sampler']['cursor']==effective*expected_updates
        assert saved['microstep']==0 and not saved['partial_accumulation']
        rows=saved['state']['metrics'];assert len(rows)==expected_updates
        assert all(row['teacher_gradients_absent'] and row['teacher_eval'] for row in rows)
        assert all(np.isfinite(row['loss']) and row['failed_attempts_total']==0 for row in rows)
        trace=[{key:r[key] for key in ('sample_ids','occurrences','augmentation_ids','mask_ids')} for r in saved['state']['traces']]
        trace_digests[kind]=hashlib.sha256(json.dumps(trace,sort_keys=True).encode()).hexdigest()
        performance=read(kind+'_performance.json')
        assert performance['successful_update_coverage_complete'] and performance['history_covered_through_update']>=expected_updates
        performances[kind]=performance
        for rec in initial['states'][kind].values():assert digest_file(rec['path'])==rec['sha256']
        if kind=='semantic':
            for name,value in saved['model'].items():
                if name.startswith('context.') and name.endswith('.omega'):
                    assert torch.equal(value,saved['model'][name.replace('context.','target.',1)])
        del saved,trace,rows
    assert len(set(trace_digests.values()))==1
    assert counts['semantic']['trainable']==counts['vit']['trainable']
    check['training_and_checkpoints']={'status':'PASS','updates_each':expected_updates,'exposures_each':effective*expected_updates,
        'trace_digests':trace_digests,'semantic_and_vit_trainable_equal':True,'timing_coverage_complete':True}
    probe_inits=set();probe_budgets=[];reconciled_predictions={};reconciliation={}
    for name,probe in evaluation['probes'].items():
        assert probe['train_examples']==10100 and probe['train_classes']==101
        assert probe['encoder_unchanged'] and probe['standardization_fit_scope']=='probe_train_only'
        assert probe['completed_epochs']==config['probe']['epochs']
        assert 1<=probe['selected_epoch']<=config['probe']['epochs']
        probe_inits.add(probe['initialization_hash']);probe_budgets.append(probe['budget'])
        predictions=read(name+'_semantic_probe.json')['probe']['predictions']
        reconciliation[name]=reconcile_frozen_probe(folder/'semantic_features'/name,
            probe['selected_probe_path'],predictions,probe,
            split['training'],split['development'],split['heldout'])
        reconciled_predictions[name]=predictions
    assert len(probe_inits)==1 and all(v==probe_budgets[0] for v in probe_budgets)
    for name,contrast in evaluation['contrasts'].items():
        first=reconciled_predictions[contrast['first']]
        second=reconciled_predictions[contrast['second']]
        assert len(first)==len(second)==2020
        independent_difference=np.mean([float(a['correct_top1'])-float(b['correct_top1'])
                                       for a,b in zip(first,second)])*100
        assert np.isclose(independent_difference,contrast['top1']['delta_percentage_points'],rtol=0,atol=1e-10), name+' point-estimate discrepancy'
        recomputed=paired_bootstrap(first,second,repeats=config['probe']['bootstrap_repeats'],seed=config['seeds']['bootstrap'])
        assert recomputed==contrast['top1'], name+' bootstrap discrepancy'
    check['probes_and_contrasts']={'status':'PASS','probes':6,'contrasts_recomputed':len(evaluation['contrasts']),
        'shared_probe_initialization':True,'cpu_prediction_reconciliation':reconciliation,
        'interval_verification':'Bootstrap intervals are recomputed with the same helper and algorithm; predictions and point differences are reconciled independently.',
        'uncertainty':'One training seed; conditional development-image intervals only.'}
    for row in compute['rows']:
        assert row['status']=='PASS', row
        assert len(row['raw_seconds'])>=10 and row['warmup']>=3
        assert row['median_seconds']>0 and row['peak_allocated_bytes']>=row['baseline_allocated_bytes']
    check['compute_evidence']={'status':'PASS','cases':len(compute['rows']), 'scope':compute['scope']}
    ratios={}
    for comparator in ('fmm','vit'):
        ratios[comparator]={
            'semantic_step_time_ratio':performances['semantic']['complete_jepa_step_seconds']['median']/performances[comparator]['complete_jepa_step_seconds']['median'],
            'semantic_peak_memory_ratio':performances['semantic']['peak_cuda_allocated_bytes']/performances[comparator]['peak_cuda_allocated_bytes']}
    outcome={'status':'PASS','run_id':run_id,'checks':check,'training_ratios':ratios,
             'dev_top1_percent':{k:100*v['metrics']['top1'] for k,v in evaluation['probes'].items()},
             'contrasts':evaluation['contrasts'],'training_parameter_counts':counts,
             'authorized_budget_seconds':clock['deadline_unix']-clock['started_unix']}
    atomic_json(folder/'independent_review.json',outcome)
    return outcome


## Reconcile the presentation

Regenerate the compute figure with shared panel limits determined after all architectures are drawn. Check every plotted point against the final limits and show measured accumulated training cost separately from the synthetic sweep. Update only the derived PNG and an idempotent report section; the frozen experiment modules and measurements remain unchanged.

In [ ]:
def refresh_semantic_presentation(root, run_id):
    """Repair derived presentation from saved evidence without changing run sources.

    This cell is reusable and tagged ahr-export. It changes only the compute PNG,
    an anchored report section, and its own axis-bound verification artifact.
    Full visual approval remains pending a separate inspection of the PNG.
    """
    import matplotlib
    matplotlib.use('Agg',force=True)
    import matplotlib.pyplot as plt
    import numpy as np
    from pathlib import Path
    import json
    from fmm_jepa.training import atomic_json
    folder=Path(root)/'artifacts/notebook_03'/run_id
    compute=json.loads((folder/'semantic_compute_benchmark.json').read_text())
    rows=[row for row in compute['rows'] if row.get('status')=='PASS']
    kinds=('vit','fmm','semantic')
    labels={'vit':'ViT','fmm':'Original FMM','semantic':'Semantic-local'}
    colors={'vit':'#2966A0','fmm':'#A26822','semantic':'#5D7146'}
    markers={'vit':'o','fmm':'s','semantic':'^'}
    scopes=('encoder_forward','full_jepa_update')
    plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False,
                         'figure.facecolor':'white','axes.titleweight':'bold'})
    fig,axes=plt.subplots(2,2,figsize=(12,8),constrained_layout=True)
    verification=[]
    for col,scope in enumerate(scopes):
        for row_index,field,scale in ((0,'median_seconds',1000),(1,'peak_allocated_bytes',1/1024**2)):
            ax=axes[row_index,col];points=[];token_values=set()
            for kind in kinds:
                selected=sorted([r for r in rows if r['scope']==scope and r['model_id']==kind],key=lambda r:r['tokens'])
                if not selected:continue
                xs=[r['tokens'] for r in selected]
                ys=[float(r[field])*scale for r in selected]
                assert all(np.isfinite(ys)) and all(y>=0 for y in ys)
                ax.plot(xs,ys,marker=markers[kind],markersize=6,linewidth=2,
                        color=colors[kind],label=labels[kind])
                points.extend(zip(xs,ys));token_values.update(xs)
            assert points, 'Cannot render a measured panel without points'
            # Set both bounds once, AFTER every architecture has been plotted.
            # Repeated set_ylim(bottom=0) inside the series loop disables
            # autoscaling after the first series and clips later larger values.
            maximum=max(y for x,y in points)
            ax.set_ylim(0,maximum*1.12 if maximum>0 else 1)
            xmin,xmax=min(token_values),max(token_values)
            padding=max((xmax-xmin)*.05,1)
            ax.set_xlim(xmin-padding,xmax+padding)
            ax.set_xticks(sorted(token_values))
            ax.grid(color='#e6e6e6');ax.set_axisbelow(True)
            ax.set_ylabel('Median synchronized time (ms)' if row_index==0 else 'Total peak CUDA allocation (MiB)')
            if row_index==1:ax.set_xlabel('Full-grid patch tokens')
            else:
                ax.set_title('Encoder forward' if scope=='encoder_forward' else 'Full JEPA update (one microbatch)')
                ax.legend(frameon=False,loc='center left')
            xbounds=ax.get_xlim();ybounds=ax.get_ylim()
            assert all(xbounds[0]<=x<=xbounds[1] and ybounds[0]<=y<=ybounds[1] for x,y in points)
            verification.append({'scope':scope,'metric':field,'points':len(points),
                'x_limits':list(xbounds),'y_limits':list(ybounds),'all_points_within_axes':True})
    fig.suptitle('Synthetic compute · identical FP32 scaffold and batch · isolated model allocations',fontsize=12)
    figure_path=folder/'semantic_compute.png'
    fig.savefig(figure_path,dpi=160);plt.close(fig)
    performance={kind:json.loads((folder/(kind+'_performance.json')).read_text()) for kind in kinds}
    config=json.loads((folder/'resolved_config.json').read_text())
    updates=json.loads((folder/'training_summary.json').read_text())['common_successful_updates']
    lines=['## Actual accumulated training cost','',
        f'All three models completed {updates} updates at microbatch {config["training"]["microbatch"]} × {config["training"]["accumulation_steps"]} accumulation steps, with effective batch {config["training"]["microbatch"]*config["training"]["accumulation_steps"]}.',
        '', '| Model | Median complete step (s) | Peak CUDA tensor allocation (GiB) |', '|---|---:|---:|']
    costs={}
    for kind in kinds:
        perf=performance[kind]
        assert perf['successful_update_coverage_complete']
        seconds=perf['complete_jepa_step_seconds']['median'];peak=perf['peak_cuda_allocated_bytes']
        assert seconds>0 and peak>0
        costs[kind]={'median_step_seconds':seconds,'peak_allocated_bytes':peak}
        lines.append(f'| {labels[kind]} | {seconds:.3f} | {peak/1024**3:.3f} |')
    candidate=costs['semantic'];old=costs['fmm'];vit=costs['vit']
    step_old=candidate['median_step_seconds']/old['median_step_seconds']
    mem_old=candidate['peak_allocated_bytes']/old['peak_allocated_bytes']
    step_vit=candidate['median_step_seconds']/vit['median_step_seconds']
    mem_vit=candidate['peak_allocated_bytes']/vit['peak_allocated_bytes']
    lines += ['',f'Semantic-local used **{100*(1-mem_old):.1f}% less peak training memory** than the original FMM, while its median step was **{100*(1-step_old):.1f}% faster**. It did not meet the predeclared 20% step-time improvement threshold.',
              '',f'Compared with ViT, semantic-local took **{100*(step_vit-1):.1f}% more time per step** and used **{mem_vit:.2f}× peak memory**.',
              '', 'These measurements cover the actual accumulated training workload, separately from the synthetic one-microbatch sweep above. Peaks include the live model, optimizer, inputs and temporaries; reserved allocator memory and the CUDA context are excluded. GiB means 1,073,741,824 bytes.']
    actual_training_table='\n'.join(lines)
    begin='<!-- BEGIN SEMANTIC_ACTUAL_TRAINING_COST -->'
    end='<!-- END SEMANTIC_ACTUAL_TRAINING_COST -->'
    report_path=folder/'semantic_results.md';report=report_path.read_text()
    section=begin+'\n'+actual_training_table+'\n'+end
    if begin in report or end in report:
        assert report.count(begin)==report.count(end)==1, 'Malformed or duplicate report anchors'
        left,remainder=report.split(begin,1)
        _,right=remainder.split(end,1)
        report=left+section+right
    else:
        anchor='![semantic quality](semantic_quality.png)'
        assert anchor in report, 'Expected report figure anchor missing'
        report=report.replace(anchor,section+'\n\n'+anchor,1)
    report_path.write_text(report)
    value={'status':'AXIS_BOUNDS_PASS_VISUAL_REVIEW_PENDING','figure':str(figure_path),
        'axis_checks':verification,'points_checked':sum(v['points'] for v in verification),
        'training_costs':costs,'training_ratios':{'semantic_to_fmm_step':step_old,
            'semantic_to_fmm_peak':mem_old,'semantic_to_vit_step':step_vit,'semantic_to_vit_peak':mem_vit},
        'visual_inspection':'PENDING_root_image_review',
        'source_compute':'semantic_compute_benchmark.json','source_training':[kind+'_performance.json' for kind in kinds],
        'frozen_source_modules_modified':False}
    atomic_json(folder/'presentation_review.json',value)
    return {'verification':value,'actual_training_markdown':actual_training_table}


In [ ]:
RUN_ID='semantic-local-pilot-002'
REVIEW=audit_semantic_run(ROOT,RUN_ID)
REVIEW

In [ ]:
from IPython.display import display,Markdown,Image
ARTIFACT=ROOT/'artifacts/notebook_03'/RUN_ID
PRESENTATION=refresh_semantic_presentation(ROOT,RUN_ID)
display(Markdown(PRESENTATION['actual_training_markdown']))
display(Markdown((ARTIFACT/'semantic_results.md').read_text()))
for name in ('semantic_quality.png','semantic_compute.png'):
    if (ARTIFACT/name).exists():display(Image(filename=str(ARTIFACT/name)))
PRESENTATION['verification']
